# Laguna transcript CSV workflow

This notebook prepares a Slurm job for the participant transcript CSV tool. It does **not** consume Laguna compute until you explicitly run the submit cell.

Use this flow for lab sessions where the heavy transcription work should run on the cluster while the notebook remains a lightweight control surface.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd()
if not (REPO / "scripts" / "laguna_submit.py").exists():
    REPO = REPO.parent

SESSION = "session_01"
CONFIG = "configs/project_template.yaml"
VENV = str(Path.home() / "behavior-pipeline" / "transcript-env")

# Keep this False until you have reviewed the generated Slurm script.
SUBMIT = False

## 1. Build the Slurm job script only

This dry run writes a script under `output/slurm/` and prints exactly what would run on Laguna. It does not call `sbatch`.

In [ ]:
cmd = [
    "python3", "scripts/laguna_submit.py",
    "--session", SESSION,
    "--workdir", str(REPO),
    "--venv", VENV,
    "--time", "04:00:00",
    "--cpus", "8",
    "--mem", "32G",
    "--gpus", "1",
    "--",
    "--config", CONFIG,
]

subprocess.run(cmd, check=True)

## 2. Submit only when ready

Run this cell only after confirming the generated script, paths, resources, and input config are correct.

In [ ]:
if SUBMIT:
    submit_cmd = cmd[:2] + ["--submit"] + cmd[2:]
    subprocess.run(submit_cmd, check=True)
else:
    print("SUBMIT is False; no Slurm job was submitted.")

## 3. Monitor jobs and outputs

Use a terminal on Laguna:

```bash
squeue -u "$USER"
ls output/slurm
ls output/session_01
```

The expected final artifact is `output/<session_id>/utterances.csv` unless you set `--out` in the transcribe arguments.